<a href="https://colab.research.google.com/github/your-org/alexpose/blob/main/experiments/multiple-sclerosis/04_progressive_finetune_ms_pd_vicreg.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 04 - Compare training budgets using validation sources

We compare the original label-free checkpoint with additional label-free training. For each checkpoint, a frozen encoder produces features and a supervised linear head learns the condition labels. The head and its scaler fit training clips only. Validation sources choose the training budget; outer test sources stay untouched.

The filename is historical. This notebook does not use class-aware VICReg. Using labels in a training loss would make that stage supervised; it would not by itself be test leakage.

In [ ]:
# --- Setup: install dependencies (Colab installs; local usually already has them) ---
import importlib, importlib.util, subprocess, sys, os

IN_COLAB = 'google.colab' in sys.modules

def _need(mod):
    return importlib.util.find_spec(mod) is None

# Light deps used by every notebook.
_pkgs = []
for mod, pip_name in [('cv2','opencv-python'), ('mediapipe','mediapipe'),
                      ('sklearn','scikit-learn'), ('pandas','pandas'),
                      ('matplotlib','matplotlib'), ('tqdm','tqdm')]:
    if _need(mod):
        _pkgs.append(pip_name)
# torch is guarded so Colab's preinstalled GPU torch is never downgraded.
if _need('torch'):
    _pkgs.append('torch')
if _pkgs:
    print('installing:', _pkgs)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *_pkgs])
else:
    print('all light dependencies already present')

In [ ]:
# --- Make `sjepa` and `ambient` importable, locally and in Colab ---
from pathlib import Path
import sys, subprocess

def _find_exp_dir():
    # Local run: this notebook sits in experiments/multiple-sclerosis.
    here = Path.cwd()
    for p in [here, *here.parents]:
        if (p / 'sjepa' / '__init__.py').exists():
            return p
    return None

EXP_DIR = _find_exp_dir()
if EXP_DIR is None:
    # Colab: clone the repo, then point at the experiment folder.
    REPO = 'https://github.com/your-org/alexpose.git'  # <-- edit to your fork
    if not Path('alexpose').exists():
        subprocess.check_call(['git', 'clone', '--depth', '1', REPO])
    EXP_DIR = Path('alexpose') / 'experiments' / 'multiple-sclerosis'

REPO_ROOT = EXP_DIR.parents[1]
for p in (str(EXP_DIR), str(REPO_ROOT)):
    if p not in sys.path:
        sys.path.insert(0, p)
print('experiment dir:', EXP_DIR)
print('repo root     :', REPO_ROOT)

In [ ]:
# --- Paths and profile (reads the root .env if python-dotenv is present) ---
import os
try:
    from dotenv import load_dotenv
    load_dotenv(REPO_ROOT / '.env')
except Exception:
    pass

VIDEO_DIR = EXP_DIR / 'video-data-full'
ARTIFACT_DIR = EXP_DIR / 'artifacts'
KEYPOINTS_DIR = ARTIFACT_DIR / 'keypoints-full'
IMAGES_DIR = EXP_DIR / 'images'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# Pick the model size profile. 'laptop' is the fast default; set SJEPA_PROFILE=gpu
# in your .env for a larger model, or SJEPA_SMOKE=1 for a near-instant test run.
os.environ.setdefault('SJEPA_PROFILE', 'laptop')
print('SJEPA_PROFILE =', os.environ['SJEPA_PROFILE'],
      '| SJEPA_SMOKE =', os.environ.get('SJEPA_SMOKE', '0'))

## Keep each source video together

We use the same frozen five-fold registry in notebooks 02–06. A source video may have several clips; each clip may yield overlapping windows. All of those relatives stay together. Each round uses about 60% of sources for training, 20% for validation, and 20% for testing. Only notebook 06 evaluates test clips.

The splitter runs on one row per source, with condition labels used to balance source counts. It never splits windows. The loader checks the full cache, reviewed exclusions, and registry checksum. A changed cache requires a new registry and new checkpoints. See [the full method](docs/11-full-data-splits.md).

In [ ]:
from IPython.display import display
import pandas as pd
from sjepa.splits import load_full_registry, partition_records, split_summary
records, registry = load_full_registry(EXP_DIR)
FOLD = 0  # teaching example; notebook 06 independently trains all five folds
train_recs, val_recs, test_recs = partition_records(records, registry, FOLD)
display(pd.DataFrame(split_summary(records, registry)))
print('usable clips:', len(records), '| excluded raw clips:', len(registry['inventory']['exclusions']))
print('registry:', registry['registry_sha256'])

In [ ]:
from sjepa.config import get_config
from sjepa.models import build_model, pick_device
from sjepa.splits import fold_run_dir, load_partition_checkpoint
from sjepa.full_experiment import train_checkpoint, embed_records, fit_probe, score_records
cfg = get_config(); device = pick_device()
RUN_DIR = fold_run_dir(EXP_DIR, registry, cfg, FOLD)
model = build_model(cfg, device=device, repaired=True)
load_partition_checkpoint(RUN_DIR / 'ssl.pt', model, cfg, registry, FOLD, 'ssl', device)

## Continue learning from the same training sources

The usual extra budget is 400 updates; smoke mode uses 2. We keep the learned model and teacher weights but start a fresh optimizer, schedule, and centering state. This is an additional training stage, not an exact resume of the earlier optimizer.

In [ ]:
MORE = 2 if cfg.profile.endswith('smoke') else 400
state = train_checkpoint(model, train_recs, cfg, registry, FOLD, 'continued',
                         MORE, device, RUN_DIR / 'continued.pt')

## Fit on training clips; compare on validation clips

Each clip gets one vector by averaging its windows and a fixed, seeded token readout. The head is logistic regression with C=1 and balanced class weights. For selection, each validation source has total weight one, divided among its clips. We choose the higher macro-F1; a tie keeps the original checkpoint. Nothing is refit on validation.

In [ ]:
validation_scores = {}
for stage in ['ssl', 'continued']:
    m = build_model(cfg, device=device, repaired=True)
    load_partition_checkpoint(RUN_DIR / f'{stage}.pt', m, cfg, registry, FOLD, stage, device)
    probe = fit_probe(embed_records(m, train_recs, cfg, device), train_recs)
    pred = probe.predict(embed_records(m, val_recs, cfg, device))
    validation_scores[stage] = score_records(val_recs, pred, equal_source=True).macro_f1
selected = 'continued' if validation_scores['continued'] > validation_scores['ssl'] else 'ssl'
print('Validation source-weighted macro-F1:', validation_scores)
print('Selected stage:', selected, '| outer test clips have not been evaluated')

## What the comparison means

This is one small validation example. Its scores help choose between two budgets and are not final performance estimates. Notebook 06 repeats the complete train-and-select procedure from fresh model weights inside each outer fold. Do not change the procedure after seeing its test results and then present those same results as an untouched test.